# Chapter 5: Risk-averse models

## Setup

In [ ]:
import os

import gurobipy as gp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gurobipy import GRB
from matplotlib.colors import to_rgb

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

os.makedirs("results", exist_ok=True)
plt.rcParams.update({"font.size": 9, "lines.linewidth": 1.2, "pdf.fonttype": 42})

k, s, r = 15, 45, 10
GROUPS = [[0, 1, 2], [3, 4, 5], [6, 7, 8, 9]]
Q_COLS = [f"Q{i}" for i in range(1, 11)]
ALPHAS = [0.01, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 1]
N, SEED, TIME_LIMIT = 1000, 8000, 3600
# the objective is bounded, so INF_OR_UNBD means infeasible
STATUS = {GRB.OPTIMAL: "optimal", GRB.INFEASIBLE: "infeasible", GRB.INF_OR_UNBD: "infeasible",
          GRB.TIME_LIMIT: "time limit"}

p_bench = np.array([0.05, 0.25, 0.15, 0.10, 0.20, 0.08, 0.12, 0.04, 0.17, 0.22])
y_bench = np.array([0.80, 0.40, 0.50, 0.70, 0.40, 0.60, 0.75, 0.90, 0.65, 0.85])
c_bench = np.array([40, 25, 35, 30, 21, 55, 45, 37, 52, 42], dtype=float)

In [ ]:
# instance l* = 3, identical to chapter 4
np.random.seed(3003)
p = np.random.uniform(np.maximum(0.02, p_bench - 0.05), np.minimum(0.35, p_bench + 0.05))
y = np.random.uniform(np.maximum(0.30, y_bench - 0.15), np.minimum(0.95, y_bench + 0.15))
c = np.random.uniform(np.maximum(r + 1, 0.90 * c_bench), np.minimum(s + k - 1, 1.10 * c_bench))
mu = np.random.uniform(350, 450)
sigma = np.random.uniform(0.20, 0.29) * mu
inst = {"p": p, "y": y, "c": c, "mu": mu, "sigma": sigma}

print(f"mu = {mu:.2f}, sigma = {sigma:.2f}")
pd.DataFrame({"supplier": range(1, 11), "p": p, "y": y, "c": c}).round(3)

## Functions

In [ ]:
# identical to chapter 4
def generate(N, seed, inst, E_fixed=None):
    rng = np.random.default_rng(seed)
    n = len(inst["p"])
    # group events and number of hit groups
    E = (rng.random((N, 3)) < 0.15).astype(int)
    if E_fixed is not None:
        E = np.tile(np.asarray(E_fixed, dtype=int), (N, 1))
    H = E.sum(axis=1)
    # disruption probabilities and yield factors
    p_mod = np.tile(inst["p"], (N, 1))
    for g, members in enumerate(GROUPS):
        hit = E[:, g] == 1
        p_mod[np.ix_(hit, members)] = np.minimum(0.90, inst["p"][members] + 0.25)
    R = np.where(rng.random((N, n)) < p_mod, inst["y"], 1.0)
    # stressed demand and spot price
    D = np.maximum(0.0, (1 + 0.1 * H) * rng.normal(inst["mu"], inst["sigma"], N))
    SP = 30 + 15 * H + 30 * rng.random(N)
    return H, R, D, SP


def profit(Q, R, D, SP, inst):
    X = R @ Q
    shortage = np.maximum(0.0, D - X)
    # each unit short is bought on the spot market or lost, whichever is cheaper
    return (s - r) * D + r * X - R @ (inst["c"] * Q) - np.minimum(s - r + k, SP - r) * shortage


def active(Q):
    return " ".join(str(i + 1) for i in np.flatnonzero(Q > 1e-6))

In [ ]:
def add_recourse(m, R, D, SP, inst):
    N, n = R.shape
    # first stage: order quantities
    Q = m.addMVar(n, lb=0.0)
    # second stage: sales, leftover, spot purchase, lost sales
    z = m.addMVar(N, lb=0.0)
    v = m.addMVar(N, lb=0.0)
    B = m.addMVar(N, lb=0.0)
    L = m.addMVar(N, lb=0.0)
    # supply + spot = sales + leftover; sales + lost sales = demand (as in the SAA LP of chapter 4)
    m.addConstr(z + v - R @ Q - B == 0)
    m.addConstr(z + L == D)
    # profit per scenario
    return Q, s * z + r * v - SP * B - k * L - (R * inst["c"]) @ Q


def risk(Q, alpha, R, D, SP, inst):
    Pi = np.sort(profit(Q, R, D, SP, inst))
    n_tail = round(alpha * len(Pi))
    # VaR = (alpha N + 1)-th smallest profit (none at alpha = 1), CVaR = mean of the alpha N worst
    VaR = Pi[n_tail] if n_tail < len(Pi) else np.nan
    return {"mean": Pi.mean(), "sd": Pi.std(ddof=1), "VaR": VaR, "CVaR": Pi[:n_tail].mean()}

In [ ]:
def solve_rn(R, D, SP, inst):
    with gp.Model(env=env) as m:
        Q, Pi = add_recourse(m, R, D, SP, inst)
        # expected profit
        m.setObjective(Pi.sum() / len(D), GRB.MAXIMIZE)
        m.optimize()
        return Q.X.copy(), m.ObjVal, m.Runtime


def solve_cvar(alpha, R, D, SP, inst):
    with gp.Model(env=env) as m:
        Q, Pi = add_recourse(m, R, D, SP, inst)
        # shortfall eta_w of the profit below t
        t = m.addVar(lb=-GRB.INFINITY)
        eta = m.addMVar(len(D), lb=0.0)
        m.addConstr(eta >= t - Pi)
        # CVaR: t minus the average shortfall over the alpha-tail
        m.setObjective(t - eta.sum() / (alpha * len(D)), GRB.MAXIMIZE)
        m.optimize()
        return Q.X.copy(), m.ObjVal, m.Runtime


def solve_var(alpha, tau, M, R, D, SP, inst, Q_start=None):
    with gp.Model(env=env) as m:
        Q, Pi = add_recourse(m, R, D, SP, inst)
        # valid inequality: orders with y @ Q > D_max are dominated in every scenario
        m.addConstr(inst["y"] @ Q <= D.max())
        if alpha < 1:
            # at most alpha N scenarios below tau (vacuous at alpha = 1)
            phi = m.addMVar(len(D), vtype=GRB.BINARY)
            m.addConstr(Pi >= tau - M * phi)
            m.addConstr(phi.sum() <= alpha * len(D))
        if Q_start is not None:
            Q.Start = Q_start
        m.setObjective(Pi.sum() / len(D), GRB.MAXIMIZE)
        m.Params.TimeLimit = TIME_LIMIT
        m.optimize()
        status = STATUS.get(m.Status, str(m.Status))
        if m.SolCount == 0:
            return None, status, np.nan, m.Runtime
        return Q.X.copy(), status, m.MIPGap if m.IsMIP else 0.0, m.Runtime

## Risk-neutral reference

In [ ]:
# one fixed scenario set for all models
_, R, D, SP = generate(N, SEED, inst)

# risk-neutral solution, with VaR and CVaR per alpha
Q_RN, z_RN, t_RN = solve_rn(R, D, SP, inst)
rn = pd.DataFrame([{"alpha": alpha, "active": active(Q_RN), **dict(zip(Q_COLS, Q_RN)),
                    **risk(Q_RN, alpha, R, D, SP, inst), "runtime_s": t_RN} for alpha in ALPHAS])
rn.round(2)

## 5.1 CVaR model

In [ ]:
Q_cvar, obj, rows = {}, {}, []
for alpha in ALPHAS:
    Q_cvar[alpha], obj[alpha], runtime = solve_cvar(alpha, R, D, SP, inst)
    rows.append({"alpha": alpha, "active": active(Q_cvar[alpha]), **dict(zip(Q_COLS, Q_cvar[alpha])),
                 **risk(Q_cvar[alpha], alpha, R, D, SP, inst), "runtime_s": runtime})
cvar = pd.DataFrame(rows)

# LP objective against the CVaR of the realized profits, and alpha = 1 against risk-neutral
print(f"max |LP objective - CVaR| = {np.abs(cvar['CVaR'] - pd.Series(obj.values())).max():.1e}")
print(f"alpha = 1: |objective - z_RN| = {abs(obj[1] - z_RN):.1e}, max |Q - Q_RN| = {np.abs(Q_cvar[1] - Q_RN).max():.1e}")
cvar.round(2)

## 5.2 VaR-constrained model

In [ ]:
# smallest valid big-M: lowest profit per scenario over {Q >= 0, y @ Q <= D_max}
# Pi_w(Q) is concave in Q, so the minimum over {Q >= 0, y @ Q <= D_max} lies in one of the 11 vertices
vertices = [np.zeros(10)] + [D.max() / y[i] * np.eye(10)[i] for i in range(10)]
Pi_low = np.min([profit(V, R, D, SP, inst) for V in vertices], axis=0)

Q_var, rows = {}, []
for alpha in ALPHAS:
    # threshold: VaR of the CVaR solution, which is also the MIP start
    tau = risk(Q_cvar[alpha], alpha, R, D, SP, inst)["VaR"]
    # binding if the risk-neutral solution violates the threshold
    binding = bool(risk(Q_RN, alpha, R, D, SP, inst)["VaR"] < tau)
    M = np.maximum(0.0, tau - Pi_low)
    Q_hat, status, gap, runtime = solve_var(alpha, tau, M, R, D, SP, inst, Q_start=Q_cvar[alpha])
    Q_var[alpha] = Q_hat
    if Q_hat is None:
        rows.append({"alpha": alpha, "tau": tau, "binding": binding, "status": status, "runtime_s": runtime})
        continue
    # scenarios below tau, with a small tolerance for scenarios at tau
    n_below = (profit(Q_hat, R, D, SP, inst) < tau - 1e-6).sum()
    rows.append({"alpha": alpha, "tau": tau, "binding": binding, "status": status, "active": active(Q_hat),
                 **dict(zip(Q_COLS, Q_hat)),
                 **risk(Q_hat, alpha, R, D, SP, inst), "n_below": n_below, "mip_gap": gap, "runtime_s": runtime})
var = pd.DataFrame(rows)

print("n_below <= alpha N in every row:", bool((var["n_below"] <= var["alpha"] * N).all()))
print(f"alpha = 1: |mean - z_RN| = {abs(var['mean'].iloc[-1] - z_RN):.1e}, "
      f"max |Q - Q_RN| = {np.abs(var[Q_COLS].iloc[-1].to_numpy(dtype=float) - Q_RN).max():.1e}")
var.round(2).assign(mip_gap=var["mip_gap"].round(6))

In [ ]:
# feasibility check: fixed threshold tau = 0.5 z_RN, without MIP start
tau_fix = 0.5 * z_RN
print(f"tau = {tau_fix:.2f}")
for alpha in [0.01, 0.1]:
    Q_hat, status, _, runtime = solve_var(alpha, tau_fix, np.maximum(0.0, tau_fix - Pi_low), R, D, SP, inst)
    n_below = "-" if Q_hat is None else (profit(Q_hat, R, D, SP, inst) < tau_fix - 1e-6).sum()
    print(f"alpha = {alpha}: {status if Q_hat is None else 'feasible'}, scenarios below tau = {n_below}, "
          f"runtime = {runtime:.1f} s")

## 5.3 Out-of-sample evaluation

In [ ]:
# independent scenario set
_, R_oos, D_oos, SP_oos = generate(100_000, 8001, inst)
taus = dict(zip(var["alpha"], var["tau"]))

rows = []
for alpha in ALPHAS:
    for model, Q in [("RN", Q_RN), ("CVaR", Q_cvar[alpha]), ("VaR", Q_var[alpha])]:
        if Q is None:
            continue
        row = {"model": model, "alpha": alpha, **risk(Q, alpha, R_oos, D_oos, SP_oos, inst)}
        # share of out-of-sample scenarios below tau for the VaR solutions
        if model == "VaR" and alpha < 1:
            row["share_below_tau"] = (profit(Q, R_oos, D_oos, SP_oos, inst) < taus[alpha]).mean()
        rows.append(row)
oos = pd.DataFrame(rows)
oos.round(2).assign(share_below_tau=oos["share_below_tau"].round(4))

## Figures

In [ ]:
# order quantities of suppliers 5 and 2 per alpha
x = np.arange(len(ALPHAS))
orders = pd.DataFrame({"alpha": ALPHAS, "Q5_CVaR": cvar["Q5"], "Q5_VaR": var["Q5"],
                       "Q2_CVaR": cvar["Q2"], "Q2_VaR": var["Q2"]})
print(f"Q5_RN = {Q_RN[4]:.2f}")
print(orders.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.axhline(Q_RN[4], color="0.5", linestyle=":", label=r"$Q_5^{\mathrm{RN}}$")
for model, table, color, style, marker in [("CVaR", cvar, "tab:blue", "-", "o"), ("VaR", var, "tab:orange", "--", "s")]:
    ax.plot(x, table["Q5"], color=color, linestyle=style, marker=marker, markersize=4, label=f"{model} model")
    ax.plot(x, table["Q2"], color=color, linestyle=style, marker=marker, markersize=4)
ax.text(0, 380, "supplier 5", ha="left", va="center")
ax.text(0, 110, "supplier 2", ha="left", va="center")
ax.set_xticks(x, [f"{a:g}" for a in ALPHAS])
ax.set_xlabel(r"$\alpha$")
ax.set_ylabel("Order quantity")
ax.legend(frameon=False)
fig.savefig("results/fig_orders_by_alpha.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# change in CVaR and expected profit relative to RN per alpha < 1
tail = ALPHAS[:-1]
x = np.arange(len(tail))
trade = pd.DataFrame({"alpha": tail})
for model, table in [("CVaR", cvar), ("VaR", var)]:
    trade[f"dCVaR_{model}"] = (table["CVaR"] - rn["CVaR"]).to_numpy()[:-1]
    trade[f"dmean_{model}"] = (table["mean"] - rn["mean"]).to_numpy()[:-1]
print(trade.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(4.5, 3))
width = 0.38
for i, (model, color) in enumerate([("CVaR", "tab:blue"), ("VaR", "tab:orange")]):
    pos = x + (i - 0.5) * width
    # lighter tint of the model color for the change in expected profit
    light = 0.5 + 0.5 * np.array(to_rgb(color))
    ax.bar(pos, trade[f"dCVaR_{model}"], width, color=color, edgecolor="white", linewidth=0.5,
           label=rf"{model} model: $\Delta\,\mathrm{{CVaR}}_{{1-\alpha}}$")
    ax.bar(pos, trade[f"dmean_{model}"], width, color=light, edgecolor="white", linewidth=0.5,
           label=f"{model} model: $\\Delta$ expected profit")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x, [f"{a:g}" for a in tail])
ax.set_xlabel(r"$\alpha$")
ax.set_ylabel("Change relative to RN")
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2))
fig.savefig("results/fig_tradeoff_by_alpha.pdf", bbox_inches="tight")
plt.show()

## Export

In [ ]:
rn.round(2).to_csv("results/5_0_risk_neutral.csv", index=False, na_rep="-")
cvar.round(2).to_csv("results/5_1_cvar.csv", index=False, na_rep="-")
var.round(2).assign(mip_gap=var["mip_gap"].round(6)).to_csv("results/5_2_var.csv", index=False, na_rep="-")
oos.round(2).assign(share_below_tau=oos["share_below_tau"].round(4)).to_csv("results/5_3_oos.csv", index=False,
                                                                            na_rep="-")